# THGS — Yolk / White 정성 진단 파이프라인 (Ramen)

**Training-Free Hierarchical Scene Understanding for Gaussian Splatting with Superpoint Graphs**

이 노트북은 **THGS_Pipeline_ramen.ipynb**의 후속 정성 진단 노트북으로,
`egg yolk` vs `egg white` 분리 실패를 **GT 없이 사람 눈으로** 검증합니다.

## 🔧 선행 조건

**`THGS_Pipeline_ramen.ipynb`를 한 번 실행**하여 다음이 이미 Google Drive에 백업되어 있어야 합니다:
- `THGS_cache/ramen_language_features/` — SAM + CLIP 피처
- `THGS_cache/ramen_model/sai_nag.pt` 등 — 파이프라인 최종 산출물

이 노트북은 **파이프라인을 재실행하지 않고** Drive에서 복원해서 바로 정성 진단에 들어갑니다.

| 단계 | 내용 |
|------|------|
| 0 | 환경 설정 (베이스라인과 100% 동일 — 의존성 / repo / submodule) |
| 1 | 데이터셋 + 2DGS 베이스 + Drive 캐시 복원 |
| Y-0 | 공통 헤더 (Scene/Gaussian/NAG/CLIP 로드) |
| Y-1 | SAM 자동 마스크 컬러 시각화 (N1/N2 정성) |
| Y-2 | `egg yolk` / `egg white` 쿼리 결과 + 겹침 진단 (메인) |
| Y-3 | Query config sweep (level/topk 6가지 정성 비교) |
| Y-4 | 사람 판정표 (수동 체크용) |


---
## Cell 0. 환경 설정 (베이스라인과 완전 동일)

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Google Drive 마운트 (모든 캐시 저장/복원에 사용)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/THGS_cache"
import os
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f"Drive 캐시: {DRIVE_CACHE}")
print(f"  기존 캐시: {os.listdir(DRIVE_CACHE) if os.listdir(DRIVE_CACHE) else '없음'}")

In [ ]:
# GitHub 레포에서 THGS 클론 + submodule 무결성 검증/복구
import os

REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"

# 1) 레포 없으면 --recursive 최초 clone
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)

# 2) 원격 변경사항 pull (conflict 방지용 autostash)
!git pull --rebase --autostash 2>&1 | tail -3

# 3) submodule URL 동기화 (.gitmodules 변경 반영)
!git submodule sync --recursive

# 4) 강제 재초기화 + fetch (--force: working tree 꼬여도 덮어씀)
!git submodule update --init --recursive --force 2>&1 | tail -20

# 5) 핵심 소스 파일 존재 여부로 실제 무결성 검증
REQUIRED_FILES = [
    "submodules/simple-knn/simple_knn.cu",
    "submodules/simple-knn/spatial.cu",
    "submodules/simple-knn/setup.py",
    "submodules/diff-surfel-rasterization/setup.py",
    "ext/spt/dependencies/FRNN/setup.py",
    "ext/spt/dependencies/parallel_cut_pursuit",
    "ext/spt/dependencies/grid_graph",
]

def _submodule_root(path):
    '''파일 경로 → submodule 루트 디렉터리'''
    parts = path.split('/')
    if parts[0] == "submodules":
        return "/".join(parts[:2])          # submodules/<name>
    if parts[0] == "ext" and parts[1] == "spt":
        return "/".join(parts[:4])          # ext/spt/dependencies/<name>
    return parts[0]

def _check_missing():
    return [p for p in REQUIRED_FILES if not os.path.exists(p)]

missing = _check_missing()

# 6) 누락된 게 있으면: 해당 submodule 디렉터리 통째 삭제 후 재클론
if missing:
    print(f"[REPAIR] 누락된 파일/폴더: {missing}")
    broken_roots = sorted({_submodule_root(p) for p in missing})
    for root in broken_roots:
        if os.path.exists(root):
            print(f"  rm -rf {root}")
            !rm -rf {root}
    print("  git submodule update --init --recursive 재실행...")
    !git submodule update --init --recursive 2>&1 | tail -10
    missing = _check_missing()
    if missing:
        raise RuntimeError(
            f"submodule 복구 실패: {missing}\n"
            "원인 후보: 네트워크 일시 장애(특히 gitlab.inria.fr), 권한, Colab DNS.\n"
            "해결: 잠시 후 이 셀을 다시 실행하거나 런타임을 해제 후 재시도."
        )

# 7) 최종 검증 리포트
print()
print("=" * 60)
print("submodule 무결성 체크")
print("=" * 60)
for p in REQUIRED_FILES:
    status = "OK " if os.path.exists(p) else "MISS"
    print(f"  [{status}] {p}")

print()
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
# 의존성 설치

# 0) NumPy 2.x와 호환되는 opencv 설치를 위해 먼저 업그레이드
!pip install -q --upgrade opencv-python

# 1) 기본 pip 패키지
!pip install -q plyfile==0.8.1 trimesh kiui pymeshlab open3d scipy \
    omegaconf open_clip_torch transformations transformers yapf \
    pycocotools mediapy lpips scikit-image einops dearpygui \
    h5py colorhash seaborn pyrootutils hydra-core hydra-colorlog \
    hydra-submitit-launcher numba pytorch-lightning rich \
    ipyfilechooser natsort gdown ninja

# 2) PyTorch Geometric (SPT 라이브러리가 PyG 2.3.0 기준 → 반드시 고정)
import torch
CUDA_VERSION = torch.version.cuda.replace(".", "")
TORCH_VERSION = torch.__version__.split("+")[0]
print(f"PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}")

!pip install -q torch_geometric==2.3.0
!pip install -q pyg_lib torch_scatter torch_cluster \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html \
    || pip install -q torch_scatter torch_cluster

# 버전 확인
import torch_geometric
print(f"torch_geometric: {torch_geometric.__version__} (2.3.0 필수)")

In [ ]:
# 3) CUDA 서브모듈 빌드
import torch, os

# 0) 사전 검증: 소스 파일이 실제 존재하는지 확인 (submodule 깨졌으면 여기서 바로 중단)
required_src = [
    "submodules/simple-knn/simple_knn.cu",
    "submodules/simple-knn/spatial.cu",
    "submodules/simple-knn/setup.py",
    "submodules/diff-surfel-rasterization/setup.py",
]
_missing = [p for p in required_src if not os.path.exists(p)]
assert not _missing, (
    f"서브모듈 소스 누락: {_missing}\n"
    f"→ Cell 4(클론 셀)를 다시 실행하세요. Cell 4의 자동 복구 로직이 처리합니다."
)

# CUDA_HOME 설정 & GPU 아키텍처 자동 감지
CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda")
os.environ["CUDA_HOME"] = CUDA_HOME
cc = torch.cuda.get_device_capability()
TORCH_CUDA_ARCH = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH_CUDA_ARCH
print(f"CUDA_HOME: {CUDA_HOME}")
print(f"GPU arch: {TORCH_CUDA_ARCH} ({torch.cuda.get_device_name(0)})")

# simple-knn 소스 패치 (CUDA 12.x / Blackwell 호환)
# 1) FLT_MAX 미정의 → #include <cfloat> 추가, 중복 #define __CUDACC__ 제거
!sed -i 's|#include "simple_knn.h"|#include "simple_knn.h"\n#include <cfloat>|' submodules/simple-knn/simple_knn.cu
!sed -i '/#define __CUDACC__/d' submodules/simple-knn/simple_knn.cu
# 2) deprecated data<float>() → data_ptr<float>()
!sed -i 's/\.data<float>()/\.data_ptr<float>()/g' submodules/simple-knn/spatial.cu

# 빌드
!pip install submodules/diff-surfel-rasterization 2>&1 | tail -3
!pip install submodules/simple-knn 2>&1 | tail -3

# 빌드 검증
try:
    from diff_surfel_rasterization import GaussianRasterizer
    print("[OK] diff-surfel-rasterization")
except Exception as e:
    print(f"[FAIL] diff-surfel-rasterization: {e}")

try:
    import simple_knn
    print("[OK] simple-knn")
except Exception as e:
    print(f"[FAIL] simple-knn: {e}")

In [ ]:
# 4) SPT 의존성 빌드
!pip install -q ext/spt/dependencies/FRNN
!pip install -q ext/spt/dependencies/FRNN/external/prefix_sum

# grid_graph + parallel_cut_pursuit C++ 확장 컴파일
!python scripts/setup_dependencies.py build_ext

In [ ]:
# 5) 추가 의존성
!pip install -q git+https://github.com/facebookresearch/pytorch3d.git
!pip install -q git+https://github.com/drprojects/point_geometric_features.git
!pip install -q git+https://github.com/minghanqin/segment-anything-langsplat.git

In [ ]:
# 설치 검증
import torch
import open_clip
import torch_geometric
from diff_surfel_rasterization import GaussianRasterizer

print("-- 설치 검증 완료 --")
print(f"  torch: {torch.__version__}")
print(f"  torch_geometric: {torch_geometric.__version__}")
print(f"  open_clip: {open_clip.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

---
## Cell 1. 데이터셋 + 2DGS 베이스 + Drive 캐시 복원

베이스라인이 이미 Drive에 백업한 `ramen_language_features/`와 `ramen_model/sai_nag.pt`를 복원합니다.
**파이프라인 재실행은 하지 않습니다** — 복원만.

> ⚠️ 이 노트북은 `THGS_Pipeline_ramen.ipynb`를 먼저 돌려 Drive에 캐시가 있어야 동작합니다.
> 캐시 경로 구조: `THGS_cache/{scene}_language_features/`, `THGS_cache/{scene}_model/`

In [ ]:
# ============================================================
# 설정: 베이스라인과 동일
# ============================================================
DATASET = "lerf"
CONFIG_FILE = f"configs/{DATASET}.yml"
SCENES = ["ramen"]

print(f"Dataset: {DATASET}")
print(f"Config: {CONFIG_FILE}")
print(f"Scenes: {SCENES}")

In [ ]:
# 데이터셋 다운로드 & 2DGS 베이스 준비 (베이스라인 Cell 12와 동일)
import os
import gdown
import zipfile
import glob
import shutil

LERF_DATA_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
LERF_DATA_ZIP = "/content/lerf_ovs.zip"
DOVS3_FOLDER_ID = "1kdV14Gu5nZX6WOPbccG7t7obP_aXkOuC"
PRETRAINED_FOLDER_ID = "1b3bXy8XENhvpWh4nLzu06UPBZEZNX6fG"

BASE_FILES = ["cfg_args", "cameras.json", "input.ply"]
BASE_DIRS = ["point_cloud"]

def copy_base_to_output(pretrained_scene_dir, output_scene_dir):
    os.makedirs(output_scene_dir, exist_ok=True)
    for f in BASE_FILES:
        src = os.path.join(pretrained_scene_dir, f)
        dst = os.path.join(output_scene_dir, f)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    for d in BASE_DIRS:
        src = os.path.join(pretrained_scene_dir, d)
        dst = os.path.join(output_scene_dir, d)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)

def download_and_extract_pretrained(pretrained_dir):
    TMP_DIR = "pretrained/_tmp"
    gdown.download_folder(id=PRETRAINED_FOLDER_ID, output=TMP_DIR, quiet=False)
    for d in [TMP_DIR] + glob.glob(f"{TMP_DIR}/*/"):
        for zf in glob.glob(f"{d}/*.zip"):
            name = os.path.splitext(os.path.basename(zf))[0]
            print(f"  {name}.zip 해제 → {pretrained_dir}/")
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(pretrained_dir)
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    for zf in glob.glob(f"{os.path.dirname(pretrained_dir)}/*.zip"):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(pretrained_dir)
        os.remove(zf)

if DATASET == "lerf":
    DATA_DIR = "data/lerf-ovs"
    PRETRAINED_DIR = "pretrained/lerf"
    OUTPUT_DIR = "output/lerf"

    # [1/3] LERF-OVS 데이터
    all_data_exist = all(os.path.exists(f"{DATA_DIR}/{sc}") for sc in SCENES)
    if not all_data_exist:
        print("[1/3] LERF-OVS 데이터셋 다운로드 중...")
        gdown.download(id=LERF_DATA_ID, output=LERF_DATA_ZIP, quiet=False)
        with zipfile.ZipFile(LERF_DATA_ZIP, 'r') as z:
            z.extractall("data/")
        os.remove(LERF_DATA_ZIP)
        if not os.path.exists(DATA_DIR) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_DIR)
        print(f"  완료: {os.listdir(DATA_DIR)}")
    else:
        print(f"[1/3] LERF-OVS 이미 존재: {os.listdir(DATA_DIR)}")

    # [2/3] 저자 제공 Pre-trained
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    all_pretrained_exist = all(
        os.path.exists(f"{PRETRAINED_DIR}/{sc}/point_cloud") for sc in SCENES
    )
    if not all_pretrained_exist:
        print(f"\n[2/3] 저자 제공 Pre-trained 다운로드 → {PRETRAINED_DIR}/")
        download_and_extract_pretrained(PRETRAINED_DIR)
    else:
        print(f"[2/3] 저자 Pre-trained 이미 존재: {os.listdir(PRETRAINED_DIR)}")

    # [3/3] 2DGS 베이스 → output/
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for scene in SCENES:
        pretrained_scene = f"{PRETRAINED_DIR}/{scene}"
        output_scene = f"{OUTPUT_DIR}/{scene}"
        ply_check = f"{output_scene}/point_cloud/iteration_30000/point_cloud.ply"
        if not os.path.exists(ply_check):
            if os.path.exists(pretrained_scene):
                print(f"[3/3] {scene}: 2DGS 베이스 → output/ 복사")
                copy_base_to_output(pretrained_scene, output_scene)
            else:
                print(f"[3/3] {scene}: pretrained 없음 - 직접 2DGS 학습 필요")
        else:
            print(f"[3/3] {scene}: output에 2DGS 베이스 이미 존재")

In [ ]:
# Drive 캐시에서 language_features + 파이프라인 산출물 복원 (베이스라인 Cell 14와 동일)
import shutil
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

# --- language_features 복원 ---
skip_encoding = True
for scene in SCENES:
    local_feat = f"{DATA_PATH}/{scene}/language_features"
    drive_feat = f"{DRIVE_CACHE}/{scene}_language_features"

    if os.path.exists(local_feat) and len(os.listdir(local_feat)) > 0:
        print(f"[feat] {scene}: 로컬에 존재")
    elif os.path.exists(drive_feat):
        print(f"[feat] {scene}: Drive에서 복원 중...")
        shutil.copytree(drive_feat, local_feat)
        print(f"  완료 ({len(os.listdir(local_feat))} files)")
    else:
        print(f"[feat] {scene}: 없음 - 베이스라인 노트북을 먼저 실행하세요")
        skip_encoding = False

# --- 파이프라인 산출물 복원 (sai_nag.pt 등) ---
PIPELINE_FILES = ["neighbor.pt", "neighbor_new.pt", "nag-l1.pt", "sai_nag.pt"]
skip_pipeline = True
for scene in SCENES:
    model_path = f"output/{SAVE_FOLDER}/{scene}"
    drive_model = f"{DRIVE_CACHE}/{scene}_model"
    os.makedirs(model_path, exist_ok=True)

    if os.path.exists(f"{model_path}/sai_nag.pt"):
        print(f"[model] {scene}: sai_nag.pt 로컬에 존재")
    elif os.path.exists(f"{drive_model}/sai_nag.pt"):
        print(f"[model] {scene}: Drive에서 산출물 복원 중...")
        for f in PIPELINE_FILES:
            src = f"{drive_model}/{f}"
            dst = f"{model_path}/{f}"
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  {f} 복원")
        print(f"  완료")
    else:
        print(f"[model] {scene}: sai_nag.pt 없음 - 베이스라인 노트북을 먼저 실행하세요")
        skip_pipeline = False

if skip_encoding and skip_pipeline:
    print("\n==> 모든 캐시 복원 완료. Y-0으로 이동.")
else:
    print("\n[ERROR] 캐시가 부족합니다. THGS_Pipeline_ramen.ipynb를 먼저 돌려 Drive에 백업하세요.")

In [ ]:
# 최종 검증: 진단 실험에 필요한 파일이 모두 준비되었는지
required = [
    f"output/{SAVE_FOLDER}/ramen/sai_nag.pt",
    f"output/{SAVE_FOLDER}/ramen/point_cloud/iteration_30000/point_cloud.ply",
    f"{DATA_PATH}/ramen/language_features",
    f"{DATA_PATH}/ramen/images",
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print("[ERROR] 다음 파일이 없습니다:")
    for p in missing: print(f"  - {p}")
    print("\n→ THGS_Pipeline_ramen.ipynb를 먼저 실행해 Drive에 캐시하세요.")
    print(f"  Drive 경로: {DRIVE_CACHE}/ramen_language_features/, {DRIVE_CACHE}/ramen_model/")
else:
    print("[OK] 진단 실험 준비 완료")
    feat_n = len(os.listdir(f"{DATA_PATH}/ramen/language_features"))
    print(f"  language_features: {feat_n} files")
    sai_mb = os.path.getsize(f"output/{SAVE_FOLDER}/ramen/sai_nag.pt") / 1024 / 1024
    print(f"  sai_nag.pt: {sai_mb:.1f} MB")

---
## Cell Y-0. 공통 헤더 (Scene / Gaussian / NAG / CLIP 로드)

In [ ]:
import os, json, time, shutil, subprocess, sys
import numpy as np
import torch
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, ".")

from omegaconf import OmegaConf
from scene import Scene, GaussianModel
from arguments import ModelParams, PipelineParams
from gaussian_renderer import render, render_point, trace
from utils.general_utils import safe_state
from utils.vlm_utils import ClipSimMeasure
from nag_data import SemanticNAG

safe_state(True)
torch.manual_seed(42); np.random.seed(42)

cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

SCENE_NAME = "ramen"
SOURCE_PATH = os.path.abspath(f"{DATA_PATH}/{SCENE_NAME}")
MODEL_PATH  = os.path.abspath(f"output/{SAVE_FOLDER}/{SCENE_NAME}")
FEAT_DIR    = f"{DATA_PATH}/{SCENE_NAME}/language_features"

YOLK_EXP_ROOT = f"output/experiments/{SCENE_NAME}_yolk_qual"
os.makedirs(f"{YOLK_EXP_ROOT}/viz", exist_ok=True)

# 평가 프레임 (베이스라인과 동일)
EVAL_FRAMES = [
    "frame_00006", "frame_00024", "frame_00060", "frame_00065",
    "frame_00081", "frame_00119", "frame_00128",
]

# Scene + Gaussian 로드 (베이스라인 Cell 32와 동일 _Args)
class _Args:
    sh_degree = 3; embedding_dim = 10; feature_dim = 512; tab_len = 300
    source_path = SOURCE_PATH; model_path = MODEL_PATH
    images = "images"; resolution = -1; white_background = False
    data_device = "cuda"; eval = False
    render_items = ['RGB','Alpha','Normal','Depth','Edge','Curvature']
    convert_SHs_python = False; compute_cov3D_python = False
    depth_ratio = 0.0; debug = False

model_args = _Args()
gaussians = GaussianModel(model_args.sh_degree, 20)
scene = Scene(model_args, gaussians, 30000, load_sem=False)
bg = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")

# sai_nag → SemanticNAG
nag_raw = torch.load(f"{MODEL_PATH}/sai_nag.pt")
snag = SemanticNAG(nag_raw["nag"], nag_raw["nag_feat"])
vlm = ClipSimMeasure(); vlm.load_model()

train_cams = scene.getTrainCameras()
cam_by_name = {c.image_name: c for c in train_cams}
for fr in EVAL_FRAMES:
    assert fr in cam_by_name, f"{fr} not in train cams"

# Pipeline config container (render에 넘길 pipe 객체)
class _Pipe:
    compute_cov3D_python = False
    convert_SHs_python = False
    depth_ratio = 0.0
    debug = False
pipe = _Pipe()

def load_image(frm):
    return cv2.imread(f"{DATA_PATH}/{SCENE_NAME}/images/{frm}.jpg")

def render_query_mask(cam, prompt, level, topk, cur_snag=None):
    """prompt → 가우시안 활성 → render → binary mask (H, W)"""
    S = cur_snag if cur_snag is not None else snag
    vlm.encode_text(prompt)
    sims = [vlm.compute_similarity(f) for f in S.feat]
    pv = S.get_related_gaussian(sims, topk=topk, level=list(level))
    pv = pv.expand(-1, 20).cuda()
    gaussians._semantics = pv
    embd = render(cam, gaussians, pipe, bg)['semantics']
    h, w = cam.image_height, cam.image_width
    return (embd.reshape(20, -1)[0] > 0.5).reshape(h, w).cpu().numpy().astype(bool)

print("YOLK_EXP_ROOT:", YOLK_EXP_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Gaussians:", gaussians.get_xyz.shape[0])
print("NAG levels:", len(nag_raw["nag"]),
      "| SP counts per level:", [int(l.max().item()+1) for l in nag_raw["nag"]])


---
## Cell Y-1. SAM 자동 마스크 컬러 시각화 (N1/N2 정성)

THGS 파이프라인이 입력으로 쓰는 SAM **automatic** mask 결과(`{frame}_s.npy`, 4 스케일)를 색깔로 칠해서 본다.

### 보는 법
- 각 스케일에서 **노른자 영역과 흰자 영역에 다른 색이 칠해져 있으면** SAM이 분리한 것 (✅).
- **같은 색**이면 collapse (= 둘이 한 마스크) → 뒤 단계가 분리할 단서 자체가 없음 (❌).
- `s` 스케일이 part-level과 가장 가까움. `default`/`l`은 큰 객체 위주.


In [ ]:
# Y-1. SAM 자동 마스크 컬러 시각화 (GT 없이)
LEVELS = ['default', 's', 'm', 'l']

fig, axes = plt.subplots(len(EVAL_FRAMES), len(LEVELS), figsize=(4*len(LEVELS), 3*len(EVAL_FRAMES)))
if len(EVAL_FRAMES) == 1: axes = axes[None, :]

for i, frm in enumerate(EVAL_FRAMES):
    seg_path = f'{FEAT_DIR}/{frm}_s.npy'
    if not os.path.exists(seg_path):
        for j in range(len(LEVELS)):
            axes[i, j].text(0.5, 0.5, f'{frm}\nseg map 없음', ha='center', va='center')
            axes[i, j].axis('off')
        continue
    seg_maps = np.load(seg_path)  # (4, H, W)
    img = load_image(frm)
    H, W = seg_maps.shape[1:]
    img_sm = cv2.resize(img, (W, H))

    for j, lvl in enumerate(LEVELS):
        sm = seg_maps[j]
        n_ids = int(sm.max()) + 1 if sm.max() >= 0 else 0
        rng = np.random.default_rng(42)
        # ID별 무작위 색 (재현성을 위해 seed 고정)
        colors = (rng.random((max(n_ids, 1) + 1, 3)) * 255).astype(np.uint8)
        sm_clip = np.clip(sm, 0, max(n_ids - 1, 0))
        colored = colors[sm_clip]
        colored[sm == -1] = 0  # -1(unassigned)은 검정
        vis = (0.4 * img_sm + 0.6 * colored).astype(np.uint8)
        axes[i, j].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        n_unique = len(np.unique(sm)) - (1 if -1 in sm else 0)
        axes[i, j].set_title(f'{frm} / [{lvl}] ({n_unique} masks)', fontsize=9)
        axes[i, j].axis('off')

plt.tight_layout()
out_path = f'{YOLK_EXP_ROOT}/viz/sam_seg_qualitative.png'
plt.savefig(out_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'saved: {out_path}')
print('\n→ 노른자/흰자에 다른 색이 칠해졌는지 눈으로 확인. 같은 색이면 SAM이 둘을 한 마스크로 합친 것.')


---
## Cell Y-2. `egg yolk` / `egg white` 쿼리 결과 + 겹침 진단 (메인)

THGS 최종 출력 — 두 쿼리를 베이스라인 config(`level=[2,3]`, `topk=3`)로 렌더링.

### 패널 구성 (5컬럼 × 7프레임)
| 컬럼 | 내용 |
|---|---|
| 1 | `"egg yolk"` 마스크를 원본 위에 cyan 오버레이 |
| 2 | `"egg yolk"` 마스크 → **검정 배경에 흰색** (모델이 고른 픽셀만) |
| 3 | `"egg white"` 마스크를 원본 위에 white 오버레이 |
| 4 | `"egg white"` 마스크 → **검정 배경에 흰색** |
| 5 | 겹침 진단 (cyan=yolk only, white=white only, 🔴 red=둘 다 활성) |

각 패널 위에 어떤 쿼리인지 표시됩니다.

`overlap_ratio = 2·|y∩w| / (|y|+|w|)` — 0에 가까울수록 분리, 0.5↑면 거의 동일 영역.


In [ ]:
# Y-2. yolk vs white 쿼리 — 컬러 오버레이 + 검정배경 흰색 마스크 + 겹침
LEVEL_BASE = [2, 3]
TOPK_BASE  = 3
QUERY_YOLK  = 'egg yolk'
QUERY_WHITE = 'egg white'

def to_white_on_black(mask_bool, H, W):
    """binary mask → 검정 배경에 흰색 (3채널 RGB)"""
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[mask_bool] = 255
    return canvas

NCOL = 5
fig, axes = plt.subplots(len(EVAL_FRAMES), NCOL, figsize=(3.2*NCOL, 3.2*len(EVAL_FRAMES)))
if len(EVAL_FRAMES) == 1: axes = axes[None, :]

overlap_ratios = []
for i, frm in enumerate(EVAL_FRAMES):
    cam = cam_by_name[frm]
    img = load_image(frm)
    H, W = img.shape[:2]

    y_mask = render_query_mask(cam, QUERY_YOLK,  LEVEL_BASE, TOPK_BASE)
    w_mask = render_query_mask(cam, QUERY_WHITE, LEVEL_BASE, TOPK_BASE)

    if (y_mask.shape[0], y_mask.shape[1]) != (H, W):
        y_mask = cv2.resize(y_mask.astype(np.uint8), (W, H), cv2.INTER_NEAREST).astype(bool)
        w_mask = cv2.resize(w_mask.astype(np.uint8), (W, H), cv2.INTER_NEAREST).astype(bool)

    overlap = y_mask & w_mask
    y_only  = y_mask & ~w_mask
    w_only  = w_mask & ~y_mask
    denom   = y_mask.sum() + w_mask.sum()
    ratio   = float(overlap.sum() * 2) / max(denom, 1)
    overlap_ratios.append((frm, ratio, int(y_mask.sum()), int(w_mask.sum()), int(overlap.sum())))

    # Col 0: yolk overlay
    vis_y = img.copy()
    vis_y[y_mask] = (0.4*vis_y[y_mask] + 0.6*np.array([0,255,255])).astype(np.uint8)
    axes[i, 0].imshow(cv2.cvtColor(vis_y, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'{frm}\nQ="{QUERY_YOLK}" overlay  (area={int(y_mask.sum())})',
                         fontsize=9, color='darkblue')
    axes[i, 0].axis('off')

    # Col 1: yolk → white-on-black
    axes[i, 1].imshow(to_white_on_black(y_mask, H, W))
    axes[i, 1].set_title(f'Q="{QUERY_YOLK}"  pure mask (white=selected)',
                         fontsize=9, color='darkblue')
    axes[i, 1].axis('off')

    # Col 2: white overlay
    vis_w = img.copy()
    vis_w[w_mask] = (0.4*vis_w[w_mask] + 0.6*np.array([255,255,255])).astype(np.uint8)
    axes[i, 2].imshow(cv2.cvtColor(vis_w, cv2.COLOR_BGR2RGB))
    axes[i, 2].set_title(f'Q="{QUERY_WHITE}" overlay  (area={int(w_mask.sum())})',
                         fontsize=9, color='darkred')
    axes[i, 2].axis('off')

    # Col 3: white → white-on-black
    axes[i, 3].imshow(to_white_on_black(w_mask, H, W))
    axes[i, 3].set_title(f'Q="{QUERY_WHITE}"  pure mask (white=selected)',
                         fontsize=9, color='darkred')
    axes[i, 3].axis('off')

    # Col 4: overlap diagnosis (3-color)
    vis_o = img.copy()
    vis_o[y_only]  = (0.3*vis_o[y_only]  + 0.7*np.array([0,255,255])).astype(np.uint8)
    vis_o[w_only]  = (0.3*vis_o[w_only]  + 0.7*np.array([255,255,255])).astype(np.uint8)
    vis_o[overlap] = (0.3*vis_o[overlap] + 0.7*np.array([0,0,255])).astype(np.uint8)  # red (BGR)
    axes[i, 4].imshow(cv2.cvtColor(vis_o, cv2.COLOR_BGR2RGB))
    title_color = 'red' if ratio > 0.3 else ('darkorange' if ratio > 0.1 else 'darkgreen')
    axes[i, 4].set_title(f'overlap diag (red=both)\nratio={ratio:.3f}',
                         fontsize=9, color=title_color)
    axes[i, 4].axis('off')

plt.tight_layout()
out_path = f'{YOLK_EXP_ROOT}/viz/yolk_white_qualitative.png'
plt.savefig(out_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'saved: {out_path}')

print('\n=== overlap ratio 요약 ===')
print(f'{"frame":<14} {"ratio":>7}  {"y_area":>8} {"w_area":>8} {"overlap":>8}')
for frm, r, ya, wa, ov in overlap_ratios:
    flag = '❌ collapse' if r > 0.3 else ('⚠ partial' if r > 0.1 else '✅ separated')
    print(f'{frm:<14} {r:>7.3f}  {ya:>8} {wa:>8} {ov:>8}  {flag}')

avg_ratio = np.mean([r for _, r, *_ in overlap_ratios])
print(f'\n평균 overlap ratio: {avg_ratio:.3f}')
if avg_ratio > 0.3:
    print('→ ❌ Collapse 우세. 두 쿼리가 같은 가우시안 그룹을 활성화. SP/merge 단계가 합친 가능성↑')
elif avg_ratio > 0.1:
    print('→ ⚠️ 부분 분리. 일부 프레임만 분리 성공. 베이스라인 config가 차선일 수 있음 → Y-3로')
else:
    print('→ ✅ 잘 분리됨. yolk/white 분리는 베이스라인에서 이미 가능.')


---
## Cell Y-3. Query config sweep (정성)

`level`, `topk` 6가지 조합 × 대표 프레임에 두 쿼리(`egg yolk`, `egg white`)를 렌더링.

### 패널 구성 (4행 × 6config 격자, 프레임당 1장)
| 행 | 내용 |
|---|---|
| 1 | `"egg yolk"` 오버레이 |
| 2 | `"egg yolk"` **검정배경 흰색 마스크** |
| 3 | `"egg white"` 오버레이 |
| 4 | `"egg white"` **검정배경 흰색 마스크** |

각 컬럼 헤더에 config + 쿼리가 표시됩니다. baseline은 `[2,3]/top3*`.

### 보는 법
- 1·2행과 3·4행이 **같은 영역**을 칠함 → 그 config는 collapse 못 풀어줌 (❌).
- 한 config라도 1·2행과 3·4행이 **다른 영역**을 칠함 → query 튜닝으로 해결 가능 (N7 범인).


In [ ]:
# Y-3. Query config sweep — 4행(yolk overlay/yolk binary/white overlay/white binary) × N config
CONFIGS = [
    ('[1]/top1',       [1],       1),
    ('[1]/top3',       [1],       3),
    ('[2]/top1',       [2],       1),
    ('[2,3]/top3*',    [2, 3],    3),   # baseline
    ('[1,2,3]/top5',   [1,2,3],   5),
    ('[1,2,3]/top10',  [1,2,3],  10),
]

REP_FRAMES = ['frame_00006', 'frame_00060']
QUERY_YOLK  = 'egg yolk'
QUERY_WHITE = 'egg white'

def to_white_on_black(mask_bool, H, W):
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[mask_bool] = 255
    return canvas

for REP in REP_FRAMES:
    cam = cam_by_name[REP]
    img = load_image(REP)
    H, W = img.shape[:2]

    fig, axes = plt.subplots(4, len(CONFIGS), figsize=(3*len(CONFIGS), 12))
    fig.suptitle(f'{REP} — query config sweep  (Q1="{QUERY_YOLK}", Q2="{QUERY_WHITE}")',
                 fontsize=13)

    for j, (label, lvl, tk) in enumerate(CONFIGS):
        try:
            y_mask = render_query_mask(cam, QUERY_YOLK,  lvl, tk)
            w_mask = render_query_mask(cam, QUERY_WHITE, lvl, tk)
        except Exception as e:
            for r in range(4):
                axes[r, j].text(0.5, 0.5, f'err: {e}', ha='center', va='center')
                axes[r, j].axis('off')
            continue

        if (y_mask.shape[0], y_mask.shape[1]) != (H, W):
            y_mask = cv2.resize(y_mask.astype(np.uint8), (W, H), cv2.INTER_NEAREST).astype(bool)
            w_mask = cv2.resize(w_mask.astype(np.uint8), (W, H), cv2.INTER_NEAREST).astype(bool)

        overlap = y_mask & w_mask
        denom = y_mask.sum() + w_mask.sum()
        ratio = float(overlap.sum() * 2) / max(denom, 1)
        title_color = 'red' if ratio > 0.3 else ('darkorange' if ratio > 0.1 else 'darkgreen')

        # Row 0: yolk overlay
        vis_y = img.copy()
        vis_y[y_mask] = (0.4*vis_y[y_mask] + 0.6*np.array([0,255,255])).astype(np.uint8)
        axes[0, j].imshow(cv2.cvtColor(vis_y, cv2.COLOR_BGR2RGB))
        axes[0, j].set_title(f'{label}\nQ="{QUERY_YOLK}" overlay  (ovl={ratio:.2f})',
                             fontsize=9, color=title_color)
        axes[0, j].axis('off')

        # Row 1: yolk binary on black
        axes[1, j].imshow(to_white_on_black(y_mask, H, W))
        axes[1, j].set_title(f'Q="{QUERY_YOLK}" pure mask', fontsize=9, color='darkblue')
        axes[1, j].axis('off')

        # Row 2: white overlay
        vis_w = img.copy()
        vis_w[w_mask] = (0.4*vis_w[w_mask] + 0.6*np.array([255,255,255])).astype(np.uint8)
        axes[2, j].imshow(cv2.cvtColor(vis_w, cv2.COLOR_BGR2RGB))
        axes[2, j].set_title(f'Q="{QUERY_WHITE}" overlay', fontsize=9, color='darkred')
        axes[2, j].axis('off')

        # Row 3: white binary on black
        axes[3, j].imshow(to_white_on_black(w_mask, H, W))
        axes[3, j].set_title(f'Q="{QUERY_WHITE}" pure mask', fontsize=9, color='darkred')
        axes[3, j].axis('off')

    plt.tight_layout()
    out_path = f'{YOLK_EXP_ROOT}/viz/query_sweep_{REP}.png'
    plt.savefig(out_path, dpi=110, bbox_inches='tight')
    plt.show()
    print(f'saved: {out_path}')

print('\n→ 1·2행과 3·4행이 모든 config에서 거의 같은 영역 → SP가 합쳐진 강한 신호 (N5/N6 범인).')
print('→ 특정 config(예: [1]/top1)에서만 분리되면 → 단순 query 튜닝으로 해결 가능 (N7 범인).')


---
## Cell Y-4. 사람 판정표 (수동 체크용)

위 시각화 결과를 보고 **직접** 표를 채운다. CSV로 저장해두면 비교/리포트에 그대로 사용 가능.

### 컬럼 의미
| 컬럼 | 체크 기준 |
|---|---|
| `sam_separated` | Y-1 시각화에서 노른자/흰자가 **다른 색**으로 칠해졌는가 (`s` 스케일 기준) |
| `yolk_correct` | Y-2의 cyan 마스크가 **노른자만** 덮는가 (흰자/그릇/면을 안 덮음) |
| `white_correct` | Y-2의 white 마스크가 **흰자만** 덮는가 |
| `collapse` | Y-2 col 2에 **빨간 영역이 절반 이상**인가 (= yolk·white가 같은 영역 활성) |

`good`/`partial`/`fail` 셋 중 하나로 채워주세요.


In [ ]:
# Y-4. 수동 판정 템플릿
import pandas as pd

# 빈 템플릿 — 사용자가 시각화 보고 직접 'good'/'partial'/'fail'로 채우기
judgment = pd.DataFrame([
    {'frame': frm,
     'sam_separated': '',     # good / partial / fail
     'yolk_correct':  '',
     'white_correct': '',
     'collapse':      '',     # good=거의 안 겹침 / partial / fail=대부분 겹침
     'note':          '',
    }
    for frm in EVAL_FRAMES
])

# 저장 (사용자가 셀 위에서 .loc[]로 값 채운 후 재실행하면 CSV로 저장)
csv_path = f'{YOLK_EXP_ROOT}/manual_judgment_template.csv'
judgment.to_csv(csv_path, index=False)
print(f'템플릿 저장: {csv_path}')
print()
print(judgment.to_string(index=False))
print()
print('=' * 60)
print('사용법:')
print("  judgment.loc[judgment.frame=='frame_00006', 'sam_separated'] = 'good'")
print("  judgment.loc[judgment.frame=='frame_00006', 'collapse']      = 'fail'")
print("  ... 모두 채운 후 → judgment.to_csv(csv_path, index=False)")
print('=' * 60)

# 정성 요약 출력 (값이 채워진 경우만)
filled = judgment[(judgment != '').any(axis=1)]
if len(filled) and (filled.drop(columns=['frame', 'note']) != '').any().any():
    print('\n=== 채워진 항목 요약 ===')
    for col in ['sam_separated', 'yolk_correct', 'white_correct', 'collapse']:
        vals = filled[col][filled[col] != '']
        if len(vals):
            counts = vals.value_counts()
            print(f'  {col:<18}: {dict(counts)}')
